In [ ]:
# ============ TEST SET TREC RUN GENERATION ============
import re, json, os
from datetime import datetime
from tqdm import tqdm
import torch
import numpy as np
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset, Dataset

# 1. Setup
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
batch_size = 16
TOP_K = 1000  # top-N ranking, TREC format'ta genelde top-1000
RUN_NAME = "TUWien-Track3-cleaning"

print(f"Device: {device}")

# 2. Cache + model
collection_embeddings = torch.load('baseline/cache/collection_embeddings.pt')
collection_keys = np.load('baseline/cache/collection_keys.npy')
print(f"Collection: {collection_embeddings.shape}")

e5_name = 'intfloat/multilingual-e5-large'
e5_tokenizer = AutoTokenizer.from_pretrained(e5_name)
e5_model = AutoModel.from_pretrained(e5_name).to(device).eval()
print("E5 loaded.")

def average_pool(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

def clean_tweet(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'^\d+/n\s*Fact\s*\d+\s*:\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^\d+/\d+\s*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def embed_texts(texts, max_len=128, desc=""):
    ds = Dataset.from_dict({"text": [f'query: {t}' for t in texts]})
    def tok(ex):
        return e5_tokenizer(ex["text"], max_length=max_len, truncation=True, padding="max_length", return_tensors="pt")
    ds = ds.map(tok, batched=True, remove_columns=ds.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
    embs = []
    with torch.inference_mode():
        for batch in tqdm(DataLoader(ds, batch_size=batch_size, num_workers=0), desc=f"Embedding {desc}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = e5_model(**batch)
            embs.append(average_pool(outputs.last_hidden_state, batch['attention_mask']).detach().cpu())
    return F.normalize(torch.vstack(embs), p=2, dim=1)

# 3. Test setlerini yükle
test_data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "test")
print(f"Test splits: {list(test_data.keys())}")
for k in test_data:
    print(f"  {k}: {len(test_data[k])} queries")

# 4. Her dil için: clean → embed → top-K → TREC dosyası yaz
os.makedirs('runs', exist_ok=True)

for lang in ['en', 'de', 'fr']:
    print(f"\n=== {lang.upper()} test ===")
    test_split = test_data[lang]
    test_indices = list(test_split["index"])
    test_texts = [clean_tweet(t) for t in test_split["text"]]

    test_emb = embed_texts(test_texts, desc=f"{lang} test")
    cos_sims = (test_emb @ collection_embeddings.T).numpy()

    # Top-K
    topk_idxs = np.argsort(-cos_sims, axis=1)[:, :TOP_K]
    topk_scores = np.take_along_axis(cos_sims, topk_idxs, axis=1)
    topk_pubkeys = collection_keys[topk_idxs]

    # TREC format: query_id Q0 doc_id rank score run_name
    out_path = f"runs/{lang}_test_run.txt"
    with open(out_path, 'w') as f:
        for i, qid in enumerate(test_indices):
            for rank in range(TOP_K):
                doc_id = topk_pubkeys[i, rank]
                score = topk_scores[i, rank]
                f.write(f"{qid} Q0 {doc_id} {rank+1} {score:.6f} {RUN_NAME}\n")
    print(f"Wrote {out_path} — {len(test_indices) * TOP_K} lines")

# 5. Birleştirilmiş tek dosya da yap (bazı submission platformları tek dosya istiyor)
combined_path = "runs/combined_test_run.txt"
with open(combined_path, 'w') as out:
    for lang in ['en', 'de', 'fr']:
        with open(f"runs/{lang}_test_run.txt") as f:
            out.write(f.read())
print(f"\nCombined: {combined_path}")

print(f"\nAll done. Submit these:")
for f in os.listdir('runs'):
    if f.endswith('.txt'):
        size_kb = os.path.getsize(f'runs/{f}') / 1024
        print(f"  runs/{f}  ({size_kb:.1f} KB)")